## All Required Librarys

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [14]:
# !pip install langchain_chroma
# !pip install beautifulsoup4

In [13]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from dotenv import load_dotenv
load_dotenv()


True

## RAG without History

In [15]:
llm =ChatGroq(model='llama-3.1-8b-instant')

In [21]:
doc_loader = WebBaseLoader('https://www.amazon.in/Grand-Theft-Auto-Standard-PlayStation/dp/B0H6X8VNQC/ref=sr_1_1?crid=1LWE68NASI66E&dib=eyJ2IjoiMSJ9.Shg3hPhBoImOKW-_q7Ss_IeDLShZpGJQNBtUGLg8YivoVKfhYiNyr4zk_sicC3iMrwNgDCFCWMnrLH1Z-Pvbg2MA4kspMgjYc-eeRLNcwMTa4B9M_W4uFw2nOhV99cfoQuw-TC6dpoHntQckef193Py95pHf-YN_QYo55hsA5gqGBpmEzEuspNunryj1HATkmEuEQ--UzNK3SMBYclVIlhY1IEr4bWJutlaBY9sAR9I.rxLNAfjpjE02lFTbjMSytiCnL59kiZOpgRucTXMcga8&dib_tag=se&keywords=ps5&qid=1783857443&sprefix=ps%2Caps%2C250&sr=8-1')

In [22]:
docs = doc_loader.load()

In [27]:
docs

[Document(metadata={'source': 'https://www.amazon.in/Grand-Theft-Auto-Standard-PlayStation/dp/B0H6X8VNQC/ref=sr_1_1?crid=1LWE68NASI66E&dib=eyJ2IjoiMSJ9.Shg3hPhBoImOKW-_q7Ss_IeDLShZpGJQNBtUGLg8YivoVKfhYiNyr4zk_sicC3iMrwNgDCFCWMnrLH1Z-Pvbg2MA4kspMgjYc-eeRLNcwMTa4B9M_W4uFw2nOhV99cfoQuw-TC6dpoHntQckef193Py95pHf-YN_QYo55hsA5gqGBpmEzEuspNunryj1HATkmEuEQ--UzNK3SMBYclVIlhY1IEr4bWJutlaBY9sAR9I.rxLNAfjpjE02lFTbjMSytiCnL59kiZOpgRucTXMcga8&dib_tag=se&keywords=ps5&qid=1783857443&sprefix=ps%2Caps%2C250&sr=8-1', 'title': 'Grand Theft Auto VI | Standard Edition | PlayStation 5 : Amazon.in: Video Games', 'description': 'Grand Theft Auto VI | Standard Edition | PlayStation 5 : Amazon.in: Video Games', 'language': 'en-in'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nGrand Theft Auto VI | Standard Edition | PlayStation 5 : Amazon.in: Video Games\n\n\n\n

In [30]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap = 200)

In [31]:
splitter = text_splitter.split_documents(docs)

In [33]:
len(splitter)

31

In [34]:
vector = Chroma.from_documents(documents=splitter,embedding=OpenAIEmbeddings())

In [35]:
retriver = vector.as_retriever()

In [36]:
retriver

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x15e295f00>, search_kwargs={})

In [37]:
systemPrompt = '''you are an helpful question answering system. 
                Answer the query from the provided context. If you don't know the answer directly say thank you and 
                I don't know the answer. Keep the answer concise. {context}'''

In [38]:
prompt = ChatPromptTemplate([('system',systemPrompt),('human','{input}')])

In [39]:
question_answer = create_stuff_documents_chain(llm,prompt)

In [40]:
rag_chain = create_retrieval_chain(retriver,question_answer)

In [47]:
Response = rag_chain.invoke({'input':'What is the price of product ?'})

In [48]:
Response

{'input': 'What is the price of product ?',
 'context': [Document(id='ec63947c-c7ff-400d-853f-bfdff1d33b07', metadata={'description': 'Grand Theft Auto VI | Standard Edition | PlayStation 5 : Amazon.in: Video Games', 'language': 'en-in', 'source': 'https://www.amazon.in/Grand-Theft-Auto-Standard-PlayStation/dp/B0H6X8VNQC/ref=sr_1_1?crid=1LWE68NASI66E&dib=eyJ2IjoiMSJ9.Shg3hPhBoImOKW-_q7Ss_IeDLShZpGJQNBtUGLg8YivoVKfhYiNyr4zk_sicC3iMrwNgDCFCWMnrLH1Z-Pvbg2MA4kspMgjYc-eeRLNcwMTa4B9M_W4uFw2nOhV99cfoQuw-TC6dpoHntQckef193Py95pHf-YN_QYo55hsA5gqGBpmEzEuspNunryj1HATkmEuEQ--UzNK3SMBYclVIlhY1IEr4bWJutlaBY9sAR9I.rxLNAfjpjE02lFTbjMSytiCnL59kiZOpgRucTXMcga8&dib_tag=se&keywords=ps5&qid=1783857443&sprefix=ps%2Caps%2C250&sr=8-1', 'title': 'Grand Theft Auto VI | Standard Edition | PlayStation 5 : Amazon.in: Video Games'}, page_content='Product description'),
  Document(id='e95664ef-2ee1-4087-953c-c87a7bdf0a44', metadata={'description': 'Grand Theft Auto VI | Standard Edition | PlayStation 5 : Amazon.in: V

In [49]:
Response = rag_chain.invoke({'input':'does the product has any cashback ?'})

In [51]:
Response['answer']

'Yes, the product has a 5% cashback offer for Prime members who use Amazon Pay ICICI Bank credit card, and 3% cashback for others.'

## RAG With History

In [78]:
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

In [79]:
system_context = '''given a chat history and the latest user question which might reference context in chat historyn the user, and don't answer without looking into the chat history'''

In [80]:
qa_prompt = ChatPromptTemplate.from_messages([('system',systemPrompt),MessagesPlaceholder('chat_history'),('human','{input} \n\n Context: \n {context}')])
cont_prompt = ChatPromptTemplate.from_messages([('system','Rephrame the query as a standalone query based on thepast history or chat')
                                                ,MessagesPlaceholder('chat_history'),('human','{input}')])

In [81]:
first_chain = create_history_aware_retriever(llm,retriver,cont_prompt)

In [82]:
QAs = create_stuff_documents_chain(llm,qa_prompt)

In [69]:
chat_history =[]

In [83]:
rag_chain = create_retrieval_chain(first_chain,QAs)

In [70]:
question = 'What is the Price od the product ?'

In [72]:
response = rag_chain.invoke({'input':question,'chat_history':chat_history})

In [74]:
response['answer']

'The price of the product is ₹5,999.00.'

In [75]:
chat_history.extend([HumanMessage(content=question),AIMessage(content=response['answer'])])

In [84]:
q2='What Question i asked you before ?'

In [85]:
response2 = rag_chain.invoke({'input':q2,'chat_history':chat_history})

In [86]:
response2['answer']

'Your question before was: "What is the Price od the product ?"'